### 1. Data reading

In [2]:
import polars as pl

dfs = []
for month in range(1, 13):
    df = pl.scan_parquet(f"./data/{month}.parquet")
    df = df.with_columns(
        pl.col("tpep_pickup_datetime").dt.cast_time_unit("ms"),
        pl.col("tpep_dropoff_datetime").dt.cast_time_unit("ms"),
    ).filter(
        (pl.col("tpep_pickup_datetime") >= pl.datetime(2024, 1, 1))
        & (pl.col("tpep_pickup_datetime") < pl.datetime(2025, 1, 1))
        & (pl.col("tpep_dropoff_datetime") <= pl.datetime(2025, 1, 1))
    )
    dfs.append(df)

df = pl.concat(dfs)

In [3]:
taxi_df = pl.scan_csv("./data/taxi_zone_lookup.csv")

In [4]:
df = df.with_columns(
    pl.col("VendorID").cast(pl.UInt8),
    pl.col("passenger_count").cast(pl.UInt8),
    pl.col("RatecodeID").cast(pl.UInt8),
    pl.col("store_and_fwd_flag").cast(pl.Categorical), # encodes repeated strings
    pl.col("PULocationID").cast(pl.UInt16),
    pl.col("DOLocationID").cast(pl.UInt16),
    pl.col("payment_type").cast(pl.UInt8),
)

df.collect_schema()



Schema([('VendorID', UInt8),
        ('tpep_pickup_datetime', Datetime(time_unit='ms', time_zone=None)),
        ('tpep_dropoff_datetime', Datetime(time_unit='ms', time_zone=None)),
        ('passenger_count', UInt8),
        ('trip_distance', Float64),
        ('RatecodeID', UInt8),
        ('store_and_fwd_flag', Categorical(ordering='physical')),
        ('PULocationID', UInt16),
        ('DOLocationID', UInt16),
        ('payment_type', UInt8),
        ('fare_amount', Float64),
        ('extra', Float64),
        ('mta_tax', Float64),
        ('tip_amount', Float64),
        ('tolls_amount', Float64),
        ('improvement_surcharge', Float64),
        ('total_amount', Float64),
        ('congestion_surcharge', Float64),
        ('Airport_fee', Float64)])

In [5]:
taxi_df.collect().describe()

statistic,LocationID,Borough,Zone,service_zone
str,f64,str,str,str
"""count""",265.0,"""265""","""265""","""265"""
"""null_count""",0.0,"""0""","""0""","""0"""
"""mean""",133.0,null,null,null
"""std""",76.643112,null,null,null
"""min""",1.0,"""Bronx""","""Allerton/Pelham Gardens""","""Airports"""
"""25%""",67.0,null,null,null
"""50%""",133.0,null,null,null
"""75%""",199.0,null,null,null
"""max""",265.0,"""Unknown""","""Yorkville West""","""Yellow Zone"""


In [6]:
taxi_df = taxi_df.with_columns(
    pl.col("LocationID").cast(pl.UInt16),
)

taxi_df.collect_schema()


Schema([('LocationID', UInt16),
        ('Borough', String),
        ('Zone', String),
        ('service_zone', String)])

### 2. Data cleaning and filtering 

In [7]:
# fill NULL values in passengers_count with 1

df = df.with_columns(
    pl.col("passenger_count").fill_null(1)
)

In [8]:
# remove rides with zero passengers

df = df.filter(pl.col("passenger_count") > 0)

In [9]:
# if a ride has over 6 passengers, replace the value with 6

df = df.with_columns(
    pl.col("passenger_count").clip(1, 6) # after previous steps the min value is 1
)

In [10]:
# remove rides lasting over 2 hours

df = df.filter(
    (pl.col("tpep_dropoff_datetime") - pl.col("tpep_pickup_datetime")) <= pl.duration(hours=2)
)

In [11]:
#  in all money-related columns, replace values with absolute value to fix negative amounts

df = df.with_columns(
    pl.col("fare_amount").abs(),
    pl.col("extra").abs(),
    pl.col("mta_tax").abs(),
    pl.col("tip_amount").abs(),
    pl.col("tolls_amount").abs(),
    pl.col("improvement_surcharge").abs(),
    pl.col("total_amount").abs(),
    pl.col("congestion_surcharge").abs(),
    pl.col("Airport_fee").abs(),
)


In [12]:
# remove rides with over 1000 dollars in any money-related column

df = df.filter(
    (pl.col("fare_amount") <= 1000)
    & (pl.col("extra") <= 1000)
    & (pl.col("mta_tax") <= 1000)
    & (pl.col("tip_amount") <= 1000)
    & (pl.col("tolls_amount") <= 1000)
    & (pl.col("improvement_surcharge") <= 1000)
    & (pl.col("total_amount") <= 1000)
    & (pl.col("congestion_surcharge") <= 1000)
    & (pl.col("Airport_fee") <= 1000)
)

In [13]:
# remove rows with values of RatecodeID or VendorID missing or outside those defined in the data dictionary

df = df.filter(
    (pl.col("RatecodeID").is_in([1, 2, 3, 4, 5, 6, 99]))
    & (pl.col("VendorID").is_in([1, 2, 6, 7]))
)


### 3. Data transformation

In [14]:
# combine payment type values for "No charge", "Dispute", "Unknown" and "Voided trip" into one type,
# so we have only credit card, cash, or other

df = df.with_columns(
    pl.when(pl.col("payment_type").is_in([0, 3, 4, 5, 6]))
    .then(pl.lit(3))
    .otherwise(pl.col("payment_type"))
    .alias("payment_type")
)


In [15]:
# replace PULocationID and DOLocationID with borough names by joining taxi zone lookup and removing unnecessary columns

df = (
    df.join(
        taxi_df.select("LocationID", pl.col("Borough").alias("PU_Borough")),
        left_on="PULocationID",
        right_on="LocationID",
        how="left",
    )
    .join(
        taxi_df.select("LocationID", pl.col("Borough").alias("DO_Borough")),
        left_on="DOLocationID",
        right_on="LocationID",
        how="left",
    )
    .drop("PULocationID", "DOLocationID")
)


In [16]:
df.limit(10).collect(streaming=True)


/tmp/ipykernel_98668/3791766996.py:1: DeprecationWarning: The old streaming engine is being deprecated and will soon be replaced by the new streaming engine. Starting Polars version 1.23.0 and until the new streaming engine is released, the old streaming engine may become less usable. For people who rely on the old streaming engine, it is suggested to pin your version to before 1.23.0.

More information on the new streaming engine: https://github.com/pola-rs/polars/issues/20947
  df.limit(10).collect(streaming=True)


VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,PU_Borough,DO_Borough
u8,datetime[ms],datetime[ms],u8,f64,u8,cat,u8,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str
2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,"""N""",2,17.7,1.0,0.5,0.0,0.0,1.0,22.7,2.5,0.0,"""Manhattan""","""Manhattan"""
1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.8,1,"""N""",1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0,"""Manhattan""","""Manhattan"""
1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.7,1,"""N""",1,23.3,3.5,0.5,3.0,0.0,1.0,31.3,2.5,0.0,"""Manhattan""","""Manhattan"""
1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.4,1,"""N""",1,10.0,3.5,0.5,2.0,0.0,1.0,17.0,2.5,0.0,"""Manhattan""","""Manhattan"""
1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.8,1,"""N""",1,7.9,3.5,0.5,3.2,0.0,1.0,16.1,2.5,0.0,"""Manhattan""","""Manhattan"""
1,2024-01-01 00:54:08,2024-01-01 01:26:31,1,4.7,1,"""N""",1,29.6,3.5,0.5,6.9,0.0,1.0,41.5,2.5,0.0,"""Manhattan""","""Manhattan"""
2,2024-01-01 00:49:44,2024-01-01 01:15:47,2,10.82,1,"""N""",1,45.7,6.0,0.5,10.0,0.0,1.0,64.95,0.0,1.75,"""Queens""","""Brooklyn"""
2,2024-01-01 00:26:01,2024-01-01 00:54:12,1,5.44,1,"""N""",2,31.0,1.0,0.5,0.0,0.0,1.0,36.0,2.5,0.0,"""Manhattan""","""Manhattan"""
2,2024-01-01 00:28:08,2024-01-01 00:29:16,1,0.04,1,"""N""",2,3.0,1.0,0.5,0.0,0.0,1.0,8.0,2.5,0.0,"""Manhattan""","""Manhattan"""


In [17]:
# add boolean variable is_airport_ride, true if there is non-zero airport fee

df = df.with_columns(
    (pl.col("Airport_fee") > 0).alias("is_airport_ride")
)

In [18]:
# add boolean variable is_rush_hour, rush hours are defined as 6:30–9:30 and 15:30-20:00 during weekdays

is_weekday = pl.col("tpep_pickup_datetime").dt.weekday().is_between(1, 5)


# I assumed that we should check intersection with rush hours
is_early_rush_hour = (
    (pl.col("tpep_dropoff_datetime").dt.time() >= pl.time(6, 30))
    & (pl.col("tpep_pickup_datetime").dt.time() < pl.time(9, 30))
    & is_weekday
)

is_late_rush_hour = (
    (pl.col("tpep_dropoff_datetime").dt.time() >= pl.time(15, 30))
    & (pl.col("tpep_pickup_datetime").dt.time() < pl.time(20, 0))
    & is_weekday
)

df = df.with_columns(
    (is_early_rush_hour | is_late_rush_hour).alias("is_rush_hour")
)


In [ ]:
df.filter(pl.col("is_rush_hour")).limit(10).collect(streaming=True)

/tmp/ipykernel_98668/4189696310.py:1: DeprecationWarning: The old streaming engine is being deprecated and will soon be replaced by the new streaming engine. Starting Polars version 1.23.0 and until the new streaming engine is released, the old streaming engine may become less usable. For people who rely on the old streaming engine, it is suggested to pin your version to before 1.23.0.

More information on the new streaming engine: https://github.com/pola-rs/polars/issues/20947
  df.filter(pl.col("is_rush_hour")).limit(10).collect(streaming=True)


### 4. Feature extraction 

In [ ]:
# apply dummy encoding to features: payment type, pickup borough, dropoff borough

df = (
    df.collect().to_dummies(["payment_type", "PU_Borough", "DO_Borough"]).lazy()
)




NameError: name 'df' is not defined